In [38]:
# Import kamping library before starting the tutorial
import kamping
import pickle


%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [39]:
gene_graphs = kamping.create_graphs('../data/metabolic_pathway', type='mixed', verbose=True)

INFO:KeggGraph:Now parsing: path:hsa00010...
INFO:KeggGraph:Graph path:hsa00010 parsed successfully!
INFO:KeggGraph:Now parsing: path:hsa01100...
INFO:KeggGraph:Graph path:hsa01100 parsed successfully!


In [40]:
gene_graph_00010 = [graph for graph in gene_graphs if graph.name == 'path:hsa00010'][0]
gene_graph_00010

KEGG Pathway: 
            [Title]: Glycolysis / Gluconeogenesis
            [Name]: path:hsa00010
            [Org]: hsa
            [Link]: https://www.kegg.jp/kegg-bin/show_pathway?hsa00010
            [Image]: https://www.kegg.jp/kegg/pathway/hsa/hsa00010.png
            [Link]: https://www.kegg.jp/kegg-bin/show_pathway?hsa00010
            Graph type: mixed 
            Number of Genes: 67
            Number of Compounds: 26
            Gene ID type : kegg
            Compound ID type : kegg
            Number of Nodes: 93
            Number of Edges: 279

In [41]:
converter = kamping.Converter('hsa', gene_target='uniprot', verbose=True)

In [42]:
for graph in gene_graphs:
    converter.convert(graph)

INFO:kamping.parser.convert:Conversion of path:hsa00010 complete!
INFO:kamping.parser.convert:Conversion of path:hsa01100 complete!


In [43]:
import pandas as pd

# uncommented code below if run the first time
# save the mols to a file
# mols.to_pickle('data/mols.pkl')
# retrieve mol from file
mols = pd.read_pickle('../data/mols.pkl')
mol_embeddings = kamping.get_mol_embeddings_from_dataframe(mols, transformer='morgan')

'
                    total 231 Invalid rows with "None" in the ROMol column


In [44]:
mol_embeddings

{'cpd:C00038': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C01180': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C20683': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C02593': array([0., 1., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C00286': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C03564': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C05452': array([0., 1., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C00603': array([0., 1., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C05443': array([0., 1., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C06157': array([0., 1., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C00055': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C04487': array([0., 1., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C05294': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C01674': array([0., 0., 0., ..., 0., 0., 0.], dtype=float32),
 'cpd:C16549': array([0., 1., 0., ..., 0., 0., 0

In [45]:
import numpy as np
from sklearn.decomposition import PCA

# Extract the embeddings and keys
keys = list(mol_embeddings.keys())
embeddings = np.array(list(mol_embeddings.values()))

# Apply PCA to reduce to 128 dimensions
pca = PCA(n_components=128)
reduced_embeddings = pca.fit_transform(embeddings)

# Create a new dictionary with the reduced embeddings
reduced_mol_embeddings = {keys[i]: reduced_embeddings[i] for i in range(len(keys))}

# Print the reduced embeddings dictionary
# print(reduced_mol_embeddings)

In [46]:
protein_embeddings = kamping.get_uniprot_protein_embeddings(gene_graphs, '../data/embedding/protein_embedding.h5') 
protein_embeddings

{'up:A0A0S2Z4B7': array([ 0.04517818,  0.08751354, -0.01246843, ..., -0.04533453,
         0.01445441,  0.0609694 ], dtype=float32),
 'up:O00750': array([ 0.00263829,  0.04652661,  0.03780524, ...,  0.00475883,
        -0.00747422, -0.00035449], dtype=float32),
 'up:O00469': array([ 0.03281771,  0.0420731 ,  0.06408788, ...,  0.01279943,
        -0.02452553, -0.02093988], dtype=float32),
 'up:O15270': array([ 0.05625306,  0.08508776,  0.04371255, ...,  0.00297614,
        -0.02832262,  0.02010833], dtype=float32),
 'up:P50213': array([0.06218641, 0.0813349 , 0.01800157, ..., 0.01593575, 0.01108014,
        0.03581247], dtype=float32),
 'up:Q8IV08': array([ 0.00572607,  0.05421366,  0.04199104, ..., -0.01639893,
         0.01633516,  0.043093  ], dtype=float32),
 'up:P51841': array([ 0.03266455,  0.05320297,  0.05699502, ...,  0.01284376,
        -0.0014937 , -0.01431572], dtype=float32),
 'up:P30838': array([ 0.0552913 ,  0.10600097,  0.04925744, ..., -0.03053111,
         0.01391122, 

In [47]:
# combine protein embeddings and metabolite embeddings into one dictionary
embeddings = {**protein_embeddings, **reduced_mol_embeddings}
len(embeddings)

2942

In [64]:
import pandas as pd
string_data = pd.read_csv("../data/string_stitch/raw/9606.protein.links.v12.0_translated.csv")
string_data = string_data[string_data['cooccurence'] > 0]
# add "up:" prefix to the protein id
string_data['protein1'] = 'up:' + string_data['protein1']
string_data['protein2'] = 'up:' + string_data['protein2']
string_data

,protein1,protein2,neighborhood,fusion,cooccurence,coexpression,experimental,database,textmining,combined_score
1502,up:Q02790,up:Q5VVH2,0,0,85,88,0,0,66,153
1692,up:Q02790,up:Q13451,0,0,63,49,0,900,730,972
1744,up:Q02790,up:Q96AY3,0,0,204,99,0,0,282,440
1813,up:Q02790,up:O95302,0,0,199,0,0,0,193,325
1909,up:Q02790,up:P26885,0,0,106,42,0,0,174,231
...,...,...,...,...,...,...,...,...,...,...
10608378,up:Q6AHZ1,up:Q8IYI6,0,0,273,49,45,0,0,282
10608403,up:A8K0Z3,up:O00329,0,0,202,0,0,0,371,476
10609159,up:P98088,up:Q9HC84,0,0,107,360,91,500,962,988
10609224,up:P98088,up:Q6W4X9,0,0,90,185,0,500,835,930


In [65]:
# use the df to create a list of tuples
additional_edges = list(string_data.itertuples(index=False, name=None))
additional_edges
# and add {'type': 'PPrel', 'subtype_name': 'additional', 'subtype_value': 'additional', 'entry1_type': 'protein', 'entry2_type': 'protein'} to each tuple
additional_edges = [(edge[0], edge[1], {'type': 'PPrel', 'subtype_name': 'additional', 'subtype_value': 'additional', 'entry1_type': 'protein', 'entry2_type': 'protein'}) for edge in additional_edges]
additional_edges

[('up:Q02790',
  'up:Q5VVH2',
  {'type': 'PPrel',
   'subtype_name': 'additional',
   'subtype_value': 'additional',
   'entry1_type': 'protein',
   'entry2_type': 'protein'}),
 ('up:Q02790',
  'up:Q13451',
  {'type': 'PPrel',
   'subtype_name': 'additional',
   'subtype_value': 'additional',
   'entry1_type': 'protein',
   'entry2_type': 'protein'}),
 ('up:Q02790',
  'up:Q96AY3',
  {'type': 'PPrel',
   'subtype_name': 'additional',
   'subtype_value': 'additional',
   'entry1_type': 'protein',
   'entry2_type': 'protein'}),
 ('up:Q02790',
  'up:O95302',
  {'type': 'PPrel',
   'subtype_name': 'additional',
   'subtype_value': 'additional',
   'entry1_type': 'protein',
   'entry2_type': 'protein'}),
 ('up:Q02790',
  'up:P26885',
  {'type': 'PPrel',
   'subtype_name': 'additional',
   'subtype_value': 'additional',
   'entry1_type': 'protein',
   'entry2_type': 'protein'}),
 ('up:Q02790',
  'up:P62942',
  {'type': 'PPrel',
   'subtype_name': 'additional',
   'subtype_value': 'additional'

In [66]:
pyg_graph, mapping = kamping.convert_to_single_pyg(gene_graphs, embeddings=embeddings,additional_edges=additional_edges, remove_edges=['PPrel'])
data= pyg_graph
data, mapping

/Users/cgu3/Documents/experiments/KAMPING/kamping/data/convert.py:324: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  hetero_data_dict[group][key] = torch.tensor(value)


(HeteroData(
   name='combined',
   type='mixed',
   compound={ x=[1001, 128] },
   gene={ x=[1510, 1024] },
   (compound, to, compound)={ edge_index=[2, 166] },
   (compound, to, gene)={ edge_index=[2, 6652] },
   (gene, to, compound)={ edge_index=[2, 6652] },
   (gene, to, gene)={ edge_index=[2, 2978] }
 ),
 {'compound': {'cpd:C00022': 0,
   'cpd:C00068': 1,
   'cpd:C00024': 2,
   'cpd:C15973': 3,
   'cpd:C00033': 4,
   'cpd:C00036': 5,
   'cpd:C16255': 6,
   'cpd:C00074': 7,
   'cpd:C00084': 8,
   'cpd:C00085': 9,
   'cpd:C00103': 10,
   'cpd:C00111': 11,
   'cpd:C00118': 12,
   'cpd:C00186': 13,
   'cpd:C00197': 14,
   'cpd:C00221': 15,
   'cpd:C00236': 16,
   'cpd:C00267': 17,
   'cpd:C00354': 18,
   'cpd:C00469': 19,
   'cpd:C00631': 20,
   'cpd:C00668': 21,
   'cpd:C01159': 22,
   'cpd:C01172': 23,
   'cpd:C05125': 24,
   'cpd:C15972': 25,
   'cpd:C00002': 26,
   'cpd:C00003': 27,
   'cpd:C00006': 28,
   'cpd:C00008': 29,
   'cpd:C00010': 30,
   'cpd:C00011': 31,
   'cpd:C00012'

In [67]:
import torch

# combine ('gene', 'to', 'compound') and ('compound', 'to', 'gene') edge_index

# switch [x, y] to [y, x]
data[('compound', 'to', 'gene')]['edge_index'] = torch.flip(data[('compound', 'to', 'gene')]['edge_index'], [0])

data[('gene', 'to', 'compound')]['edge_index'] = torch.cat([data[('gene', 'to', 'compound')]['edge_index'], data[('compound', 'to', 'gene')]['edge_index']], dim=1)

del data[('compound', 'to', 'gene')]

In [68]:
data

HeteroData(
  name='combined',
  type='mixed',
  compound={ x=[1001, 128] },
  gene={ x=[1510, 1024] },
  (compound, to, compound)={ edge_index=[2, 166] },
  (gene, to, compound)={ edge_index=[2, 13304] },
  (gene, to, gene)={ edge_index=[2, 2978] }
)

In [62]:
pyg_graph_original,original_mapping = kamping.convert_to_single_pyg(gene_graphs, embeddings=embeddings)
pyg_graph_original

HeteroData(
  name='combined',
  type='mixed',
  compound={ x=[1001, 128] },
  gene={ x=[1510, 1024] },
  (compound, to, compound)={ edge_index=[2, 166] },
  (compound, to, gene)={ edge_index=[2, 6652] },
  (gene, to, compound)={ edge_index=[2, 6652] }
)

In [54]:
pyg_graph_original['gene', 'to', 'gene']
original_mapping
# revert the mapping
reversed_mapping = {v: k for k, v in mapping['gene'].items()}

In [55]:
# get the name of the nodes
protein1 = pyg_graph_original['gene', 'to', 'gene']['edge_index'][0].tolist()
protein2 = pyg_graph_original['gene', 'to', 'gene']['edge_index'][1].tolist()
protein1 = [reversed_mapping[i] for i in protein1]
protein2 = [reversed_mapping[i] for i in protein2]
# reversed_mapping[pyg_graph_original['gene', 'to', 'gene'][0]]

KeyError: 'edge_index'

In [ ]:
# get their index in the new mapping
protein1_index = [mapping['gene'][i] for i in protein1]
protein2_index = [mapping['gene'][i] for i in protein2]

In [63]:
import torch
# create the new edge index
edge_index = torch.tensor([protein1_index, protein2_index])
# save edge_index using pickle
with open('../data/edge_index_cooccurence.pkl', 'wb') as f:
    pickle.dump(edge_index, f)

NameError: name 'protein1_index' is not defined

# You can save the data as pickle file for later use

In [70]:

# save the data as pickle file
with open('../data/pyg_graph_with_string_data_cooccurence.pkl', 'wb') as f:
    pickle.dump(data, f)

In [ ]:
import torch_geometric.transforms as T
transform = T.RandomLinkSplit(
    num_val=0.1,
    num_test=0.1,
    is_undirected=True,
    # disjoint_train_ratio=0.3, # TODO
    neg_sampling_ratio=1.0, # TODO
    add_negative_train_samples=False,
    edge_types=("gene", "to", "gene")
)
train_data, val_data, test_data = transform(data)

In [ ]:
val_data

In [ ]:
import torch

In [ ]:
from sklearn.metrics import roc_auc_score